In [10]:
# ============================================================================
# IMPORTACIÓN DE LIBRERÍAS Y CONFIGURACIÓN INICIAL
# ============================================================================

import pyvisa

rm = pyvisa.ResourceManager()

# ============================================================================
# DETECCIÓN DE INSTRUMENTOS CONECTADOS
# ============================================================================

print("=== DETECCIÓN DE INSTRUMENTOS ===")
instrumentos = rm.list_resources()
print("Instrumentos encontrados:", instrumentos)

=== DETECCIÓN DE INSTRUMENTOS ===
Instrumentos encontrados: ('ASRL4::INSTR',)


In [12]:
# ============================================================================
# IDENTIFICACIÓN DEL INSTRUMENTO
# ============================================================================

print("\n=== IDENTIFICACIÓN DEL INSTRUMENTO ===")

try:
    instrumento = rm.open_resource('ASRL4::INSTR')
    ID = instrumento.query('*IDN?')
    print("ID del instrumento:", ID)
    instrumento.close()
    
except Exception as e:
    print("Error al conectar con el instrumento:", e)


=== IDENTIFICACIÓN DEL INSTRUMENTO ===
ID del instrumento: GW,GDS-1102A-U,GES170725,V1.14



In [14]:
# ============================================================================
# ADQUISICIÓN DE DATOS DE FORMA DE ONDA
# ============================================================================
print("\n=== ADQUISICIÓN DE DATOS DE FORMA DE ONDA ===")

try:
    # Abrir conexión con el instrumento seleccionado.
    instrumento = rm.open_resource('ASRL4::INSTR')
    
    # :acquire<X>:memory? es un comando estándar SCPI.
    # Devuelve los datos totales de la forma de onda en la memoria de adquisición.
    datos = instrumento.query(':acquire1:memory?')
    x = datos.encode()
    # Abre el archivo en modo escritura
    with open("datos.txt", "w") as archivo:
        # Escribe variables usando f-strings y añade un salto de línea (\n)
        archivo.write(f"{x}")

# Al salir del bloque 'with', el archivo se cierra automáticamente
print("Se ha escrito el contenido en datos.txt")

    # Cerrar la conexión para liberar recursos.
    instrumento.close()
    
except Exception as e:
    print("Error al consultar configuración:", e)

SyntaxError: expected 'except' or 'finally' block (2840346686.py, line 20)

In [25]:
# ============================================================================
# LIMPIEZA FINAL
# ============================================================================

rm.close()
print("\n=== OPERACIÓN COMPLETADA ===")
print("Conexiones cerradas correctamente")


=== OPERACIÓN COMPLETADA ===
Conexiones cerradas correctamente


In [26]:
import pyvisa
import struct
import numpy as np

# Conectar al osciloscopio
rm = pyvisa.ResourceManager()
instr = rm.open_resource('ASRL4::INSTR')  # Cambiar por tu recurso
instr.timeout = 10000

# Extraer datos del Canal 1 (memoria normal - 4000 puntos)
print("Extrayendo datos...")
instr.write_raw(b':acquire2:memory?\n')
raw_data = instr.read_raw()

# Abre el archivo en modo escritura
with open("datos1.txt", "w") as archivo:
    # Escribe variables usando f-strings y añade un salto de línea (\n)
    archivo.write(f"{raw_data}.\n")

# Al salir del bloque 'with', el archivo se cierra automáticamente
print("Se ha escrito el contenido en datos1.txt")

# Parsear datos binarios
# Formato: #ABCDEF
# A = número de dígitos del tamaño
size_digits = int(chr(raw_data[1]))

# B = tamaño de datos  
data_size = int(raw_data[2:2+size_digits].decode())

# Posición donde empiezan los datos útiles
header_start = 2 + size_digits

# C = intervalo de tiempo (4 bytes, little-endian float)
time_interval = struct.unpack('<f', raw_data[header_start:header_start+4])[0]

# D = canal (1 byte) - saltar
# E = reservado (3 bytes) - saltar
# F = datos de forma de onda (empieza en header_start + 8)
waveform_start = header_start + 8
waveform_data = raw_data[waveform_start:]

# Convertir datos de 16 bits a valores
num_points = len(waveform_data) // 2
voltage_values = []

for i in range(num_points):
    # Cada punto son 2 bytes, MSB primero
    high_byte = waveform_data[i*2]
    low_byte = waveform_data[i*2 + 1]
    value = (high_byte << 8) | low_byte
    
    # Convertir a signed de 16 bits
    if value > 32767:
        value -= 65536
    
    voltage_values.append(value)

# Crear arrays de tiempo y voltaje
time_array = np.arange(num_points) * time_interval
voltage_array = np.array(voltage_values) * (10.0 / 32768.0)  # Escalado básico

# Abre el archivo en modo escritura
with open("datos2.txt", "w") as archivo:
    # Escribe variables usando f-strings y añade un salto de línea (\n)
    archivo.write(f"{time_array}.\n")
print("Se ha escrito el contenido en datos2.txt")

with open("datos3.txt", "w") as archivo:
    # Escribe variables usando f-strings y añade un salto de línea (\n)
    archivo.write(f"{voltage_array}.\n")
print("Se ha escrito el contenido en datos3.txt")

print(f"Puntos obtenidos: {num_points}")
print(f"Intervalo de tiempo: {time_interval} s")
print(f"Primeros 10 valores de voltaje: {voltage_array[:10]}")

# Cerrar conexión
instr.close()
rm.close()

Extrayendo datos...


VisaIOError: VI_ERROR_TMO (-1073807339): Timeout expired before operation completed.